## Experiments for constraining GPT-style model

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import spacy
# we check if any huggingface authentication is already present
import os
if "HUGGINGFACE_HUB_TOKEN" in os.environ:
    print("Huggingface authentication token found.")
else:
    print("No Huggingface authentication token found.")

2025-11-04 16:04:28.602909: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


No Huggingface authentication token found.


In [3]:
nlp = spacy.load("nl_core_news_md")
model = AutoModelForCausalLM.from_pretrained("GroNLP/gpt2-small-dutch")
tokenizer = AutoTokenizer.from_pretrained("GroNLP/gpt2-small-dutch")

In [4]:
testsent = "The quick brown fox jumps over the lazy dog."
testsent_nl = "De snelle bruine vos springt over de luie hond."
inputs = tokenizer(testsent, return_tensors="pt")
inputs_nl = tokenizer(testsent_nl, return_tensors="pt")
print(inputs_nl)

{'input_ids': tensor([[ 1023,  5643,  7178, 22075, 13171,   302,   177, 28764,  3571,    16]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [5]:
# we use the mapping of tokens to subwords to check how it is tokenized
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
tokens_nl = tokenizer.convert_ids_to_tokens(inputs_nl["input_ids"][0])
print(tokens)
print("nl, ", tokens_nl)

['The', 'Ġqu', 'ick', 'Ġbro', 'wn', 'Ġf', 'ox', 'Ġj', 'umps', 'Ġover', 'Ġthe', 'Ġl', 'az', 'y', 'Ġdog', '.']
nl,  ['De', 'Ġsnelle', 'Ġbruine', 'Ġvos', 'Ġspringt', 'Ġover', 'Ġde', 'Ġluie', 'Ġhond', '.']


In [6]:
import time
tic = time.time()
doc = nlp(testsent_nl)
toc = time.time()
print("nl parsing time:", toc - tic)
for sent in doc.sents:
    root = sent.root
    print(root.lemma_)

nl parsing time: 0.19000005722045898
springen


In [7]:
# we will predict next word, check if the sentence has a verb, if yes revert to last state, and insert the target verb instead
target_verb = "roeit"
half_sent = "De man is"
toks = tokenizer(half_sent, return_tensors="pt")
last_non_verb_state = None
# we extract model logits
output = model(**toks)
logits = output.logits
print(toks)
next_token_id = torch.argmax(logits[0, -1, :]).unsqueeze(0).unsqueeze(0)
print(next_token_id)
next_token = tokenizer.convert_ids_to_tokens(next_token_id[0][0].item())
print("Predicted next token:", next_token)
# Append the predicted token to the input
toks = {
    'input_ids': torch.cat([toks['input_ids'], next_token_id], dim=-1),
    'attention_mask': torch.cat([toks['attention_mask'], torch.tensor([[1]])], dim=-1)
}
toks

{'input_ids': tensor([[1023,  483,  250]]), 'attention_mask': tensor([[1, 1, 1]])}
tensor([[197]])
Predicted next token: Ġeen


{'input_ids': tensor([[1023,  483,  250,  197]]),
 'attention_mask': tensor([[1, 1, 1, 1]])}

In [51]:
def insert_word(word, toks):
    word_toks = tokenizer(word, return_tensors="pt")['input_ids'][0]
    # we build new input_ids and attention_mask
    new_input_ids = torch.cat([toks['input_ids'], word_toks.unsqueeze(0)], dim=-1)
    new_attention_mask = torch.cat([toks['attention_mask'], torch.ones((1, word_toks.size(0)), dtype=torch.long)], dim=-1)
    return {
        'input_ids': new_input_ids,
        'attention_mask': new_attention_mask
    }

def pick_word(words, toks):
    # we pick the word with highest score
    output = model(**toks)
    logits = output.logits
    next_token_logits = logits[0, -1, :]

    best_word = None
    word_toks = None
    best_score = float('-inf')
    for word in words:
        word_token_id = tokenizer(word, return_tensors="pt")['input_ids'][0]
        score = next_token_logits[word_token_id[0].item()].item()
        if score > best_score:
            best_score = score
            word_toks = word_token_id
            best_word = word
    print(f"Picked word: {best_word} with score {best_score}")
    # we build new input_ids and attention_mask
    new_input_ids = torch.cat([toks['input_ids'], word_toks.unsqueeze(0)], dim=-1)
    new_attention_mask = torch.cat([toks['attention_mask'], torch.ones((1, word_toks.size(0)), dtype=torch.long)], dim=-1)
    return {
        'input_ids': new_input_ids,
        'attention_mask': new_attention_mask
    }


def generate_with_word(model, tokenizer, input_text, target_verb, max_length=20, repeat=False):
    def clone_state(state):
        return {
            'input_ids': state['input_ids'].clone(),
            'attention_mask': state['attention_mask'].clone()
        }

    toks = tokenizer(input_text, return_tensors="pt")

    saw_verb_this_sent = False
    injected_verb_this_sent = False

    for _ in range(max_length):
        prev_state = clone_state(toks)

        # model forward pass
        output = model(**toks)
        logits = output.logits
        next_token_id_old = torch.argmax(logits[0, -1, :]).unsqueeze(0).unsqueeze(0)
        # or better
        next_token_id = torch.argmax(logits[0, -1, :]).view(1, 1)
        if not next_token_id_old == next_token_id:
            print("Different methods of getting next token id!")
            print("old", next_token_id_old)
            print("new", next_token_id)
        # Append the predicted token to the input
        toks = {
            'input_ids': torch.cat([toks['input_ids'], next_token_id], dim=-1),
            'attention_mask': torch.cat([toks['attention_mask'], torch.tensor([[1]])], dim=-1)
        }

        # Decode the current sequence
        current_text = tokenizer.decode(
            toks['input_ids'][0],
            skip_special_tokens=False
        )

        doc = nlp(current_text)
        last_sent = list(doc.sents)[-1]
        # we check if there has been a verb in the sentence so far
        has_verb = any(token.pos_ == "VERB" for token in last_sent)

        if has_verb and not injected_verb_this_sent:
            # we revert to last pre-token state
            toks = prev_state

            toks = insert_word(target_verb, toks)

            injected_verb_this_sent = True
            saw_verb_this_sent = True
        else:
            saw_verb_this_sent = has_verb or saw_verb_this_sent

        # we reset at sentence boundary
        if current_text.rstrip().endswith(('.', '!', '?')) and repeat:
            saw_verb_this_sent = False
            injected_verb_this_sent = False

    return tokenizer.decode(toks['input_ids'][0], skip_special_tokens=False)

# generate_with_word(model, tokenizer, "Toen ging ik naar de bakker en", " fiets", max_length=50, repeat=False)

In [64]:
DUTCH_IRREGULAR = {
    "zijn": [
        "ben", "bent", "is", "zijn",
        "was", "waren",
        "geweest"
    ],
    "hebben": [
        "heb", "hebt", "heeft", "hebben",
        "had", "hadden",
        "gehad"
    ],
    "kunnen": [
        "kan", "kunt", "kunnen",
        "kon", "konden",
        "gekund"
    ],
    "mogen": [
        "mag", "mogen",
        "mocht", "mochten",
        "gemogen"
    ],
    "willen": [
        "wil", "wilt", "willen",
        "wilde", "wilden", "wou", "wouden",
        "gewild"
    ],
    "gaan": [
        "ga", "gaat", "gaan",
        "ging", "gingen",
        "gegaan"
    ],
    "komen": [
        "kom", "komt", "komen",
        "kwam", "kwamen",
        "gekomen"
    ],
    "lopen": [
        "loop", "loopt", "lopen",
        "liep", "liepen",
        "gelopen"
    ],
    "vinden": [
        "vind", "vindt", "vinden",
        "vond", "vonden",
        "gevonden"
    ],
    "werken": [
        # regular, but we can still include common forms
        "werk", "werkt", "werken",
        "werkte", "werkten",
        "gewerkt"
    ],
    # ... breid dit woordenboek uit zoveel je wilt
}

STEMLOOS = set(list("tkfschpx"))  # ruimer dan 't kofschip, voeg x erbij voor veiligheid

def dutch_regular_inflections(verb_inf):
    forms = set()

    # 1. basisvormen
    forms.add(verb_inf)

    if verb_inf.endswith("en") and len(verb_inf) > 2:
        stam = verb_inf[:-2]
    else:
        # fallback: neem hele woord ook als stam (voor veiligheid)
        stam = verb_inf

    forms.add(stam)           # ik / imperatief
    forms.add(stam + "t")     # jij/hij/zij in tt
    forms.add(stam + "en")    # wij/jullie/zij in tt

    # 2. verleden tijd zwak
    last_char = stam[-1].lower() if len(stam) > 0 else ""
    use_te = last_char in STEMLOOS

    if use_te:
        past_sg = stam + "te"
        past_pl = stam + "ten"
        pp_ge = "ge" + stam + "t"
    else:
        past_sg = stam + "de"
        past_pl = stam + "den"
        pp_ge = "ge" + stam + "d"

    forms.add(past_sg)
    forms.add(past_pl)

    # voltooid deelwoord (ge+stam+t/d)
    forms.add(pp_ge)

    # 3. redundante/ruisvormen voor recall
    # soms ontbreekt ge- bij prefixwerkwoorden ("ontvangen" -> "ontvangen", niet "geontvangd")
    # dus voeg "stam+t" en "stam+d" ook zonder ge toe
    forms.add(stam + "t")
    forms.add(stam + "d")
    forms.add(stam + "dt")  # hypercorrect/ruis
    forms.add("ge" + stam + "te")
    forms.add("ge" + stam + "de")

    return forms

def get_dutch_inflections(verb_inf):
    verb_inf_l = verb_inf.lower()

    forms = set()

    # 1. regelmatige vormen altijd meenemen
    forms |= dutch_regular_inflections(verb_inf_l)

    # 2. als onregelmatig bekend -> voeg ALLES toe
    if verb_inf_l in DUTCH_IRREGULAR:
        for f in DUTCH_IRREGULAR[verb_inf_l]:
            forms.add(f)
        # plus ook zwakke-achtige nepvormen voor die stam
        # bv "gaan" -> "ga", "gaat", "ging", "gegaan", maar ook "ga(de)", "ga(t)" enz.
        # pak de stam volgens onze simpele regel
        if verb_inf_l.endswith("en") and len(verb_inf_l) > 2:
            stam = verb_inf_l[:-2]
        else:
            stam = verb_inf_l
        forms.add(stam + "de")
        forms.add(stam + "den")
        forms.add("ge" + stam + "d")
        forms.add(stam + "te")
        forms.add(stam + "ten")
        forms.add("ge" + stam + "t")
    # we return a list with all forms, and the forms preceded by a space
    spaced_forms = set()
    for f in forms:
        spaced_forms.add(" " + f)
    forms |= spaced_forms
    return forms

def pick_word_inflection(words, toks):
    # we pick the word with highest score
    output = model(**toks)
    logits = output.logits
    next_token_logits = logits[0, -1, :]
    best_word = None
    word_toks = None
    best_score = float('-inf')
    for word in words:
        inflections = get_dutch_inflections(word)
        for word in inflections:
            word_token_id = tokenizer(word, return_tensors="pt")['input_ids'][0]
            score = next_token_logits[word_token_id[0].item()].item()
            if score > best_score:
                best_score = score
                word_toks = word_token_id
                best_word = word
    print(f"Picked word: {best_word} with score {best_score}")
    # we build new input_ids and attention_mask
    new_input_ids = torch.cat([toks['input_ids'], word_toks.unsqueeze(0)], dim=-1)
    new_attention_mask = torch.cat([toks['attention_mask'], torch.ones((1, word_toks.size(0)), dtype=torch.long)], dim=-1)
    return {
        'input_ids': new_input_ids,
        'attention_mask': new_attention_mask
    }


def generate_with_selection(model, tokenizer, input_text, target_words, max_length=20, repeat=False):
    def clone_state(state):
        return {
            'input_ids': state['input_ids'].clone(),
            'attention_mask': state['attention_mask'].clone()
        }

    toks = tokenizer(input_text, return_tensors="pt")

    saw_verb_this_sent = False
    injected_verb_this_sent = False

    for _ in range(max_length):
        prev_state = clone_state(toks)

        # model forward pass
        output = model(**toks)
        logits = output.logits
        next_token_id = torch.argmax(logits[0, -1, :]).view(1, 1)

        # experimenting with repetition penalty and temperature
        logits_step = logits[0, -1, :].clone()
        # (optional) repetition penalty
        for token_id in toks['input_ids'][0]:
            logits_step[token_id] /= 1.2  # >1.0 discourages repeats

        # temperature
        temperature = 0.7
        logits_step = logits_step / temperature
        # top-k filter
        k = 50
        topk_vals, topk_idx = torch.topk(logits_step, k)
        probs = torch.softmax(topk_vals, dim=-1)
        choice_idx = torch.multinomial(probs, num_samples=1)
        next_token_id = topk_idx[choice_idx].view(1,1)


        # Append the predicted token to the input
        toks = {
            'input_ids': torch.cat([toks['input_ids'], next_token_id], dim=-1),
            'attention_mask': torch.cat([toks['attention_mask'], torch.tensor([[1]])], dim=-1)
        }

        # Decode the current sequence
        current_text = tokenizer.decode(
            toks['input_ids'][0],
            skip_special_tokens=False
        )

        doc = nlp(current_text)
        last_sent = list(doc.sents)[-1]
        # we check if there has been a verb in the sentence so far
        has_verb = any(token.pos_ == "VERB" for token in last_sent)

        if has_verb and not injected_verb_this_sent:
            # we revert to last pre-token state
            toks = prev_state
            toks = pick_word_inflection(target_words, toks)

            injected_verb_this_sent = True
            saw_verb_this_sent = True
        else:
            saw_verb_this_sent = has_verb or saw_verb_this_sent

        # we reset at sentence boundary
        if current_text.rstrip().endswith(('.', '!', '?')) and repeat:
            saw_verb_this_sent = False
            injected_verb_this_sent = False

    return tokenizer.decode(toks['input_ids'][0], skip_special_tokens=False)



In [65]:
print(generate_with_selection(model, tokenizer, "Toen ging ik naar de bakker en", ["fietsen", "roeien", "lopen"], max_length=50, repeat=True))

Picked word:  liep with score 87.88483428955078
Picked word:  geroeid with score 150.530517578125
Toen ging ik naar de bakker en liep door. Ik had geroeid met mijn zwarte haar, maar dat was niet zo groot als het zou zijn om een beetje van die kleur te maken.'
'Ik dacht altijd: hoe kom je in deze jurk?' vroeg hij terwijl ze even na hem
